# 02. Feature Engineering

In this notebook, we preprocess both datasets:
1. **NHANES**: We will predict Glycohemoglobin (`LBXGH`) as a continuous target (Regression) and extract lifestyle/clinical features.
2. **UCI Diabetes**: We will predict Hospital Readmission (`readmitted`) as a classification target. Crucially, we split by `patient_nbr` to avoid data leakage.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 1. Feature Engineering: NHANES

In [2]:
data_dir = '../data'

# Load merged NHANES from a saved intermediate file or re-merge
nhanes_files = ['DEMO_J.xpt', 'GHB_J.xpt', 'GLU_J.xpt', 'BIOPRO_J.xpt', 'BMX_J.xpt', 'DIQ_J.xpt', 'PAQ_J.xpt', 'SMQ_J.xpt']
nhanes_df = None
for file in nhanes_files:
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        df = pd.read_sas(file_path)
        nhanes_df = df if nhanes_df is None else pd.merge(nhanes_df, df, on='SEQN', how='outer')

# Define Target
nhanes_df = nhanes_df.dropna(subset=['LBXGH']) # Drop rows without our target (HbA1c)
y_nhanes = nhanes_df['LBXGH']

# Select a subset of features for the Digital Twin
# RIDAGEYR: Age, RIAGENDR: Gender, BMXBMI: BMI, BMXWAIST: Waist Circumference
# LBXGLU: Fasting Glucose, PAQ605: Vigorous work activity, SMQ020: Smoked 100 cigarettes in life
features = ['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'BMXWAIST', 'LBXGLU', 'PAQ605', 'SMQ020']
X_nhanes = nhanes_df[features].copy()

X_train_nh, X_test_nh, y_train_nh, y_test_nh = train_test_split(X_nhanes, y_nhanes, test_size=0.2, random_state=42)

num_features = ['RIDAGEYR', 'BMXBMI', 'BMXWAIST', 'LBXGLU']
cat_features = ['RIAGENDR', 'PAQ605', 'SMQ020']

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

nhanes_preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

X_train_nh_prep = nhanes_preprocessor.fit_transform(X_train_nh)
X_test_nh_prep = nhanes_preprocessor.transform(X_test_nh)

print(f"NHANES Processed Train Shape: {X_train_nh_prep.shape}")
print(f"NHANES Processed Test Shape: {X_test_nh_prep.shape}")

NHANES Processed Train Shape: (4836, 11)
NHANES Processed Test Shape: (1209, 11)


## 2. Feature Engineering: UCI Diabetes Dataset

We group by `patient_nbr` to ensure no data leakage across the train/test split.

In [3]:
uci_path = os.path.join(data_dir, 'diabetic_data.csv')
uci_df = pd.read_csv(uci_path, na_values='?')

# Binary target: Readmitted (Yes if <30 or >30, No otherwise)
uci_df['target'] = (uci_df['readmitted'] != 'NO').astype(int)

# Split by patient_nbr
unique_patients = uci_df['patient_nbr'].unique()
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_uci = uci_df[uci_df['patient_nbr'].isin(train_patients)].copy()
test_uci = uci_df[uci_df['patient_nbr'].isin(test_patients)].copy()

y_train_uci = train_uci['target']
y_test_uci = test_uci['target']

uci_features = ['age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 
                'num_medications', 'number_outpatient', 'number_emergency', 
                'number_inpatient', 'number_diagnoses', 'diabetesMed']

X_train_uci = train_uci[uci_features]
X_test_uci = test_uci[uci_features]

uci_num_features = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 
                    'num_medications', 'number_outpatient', 'number_emergency', 
                    'number_inpatient', 'number_diagnoses']
uci_cat_features = ['age', 'diabetesMed']

uci_preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, uci_num_features),
    ('cat', cat_transformer, uci_cat_features)
])

X_train_uci_prep = uci_preprocessor.fit_transform(X_train_uci)
X_test_uci_prep = uci_preprocessor.transform(X_test_uci)

print(f"UCI Processed Train Shape: {X_train_uci_prep.shape}")
print(f"UCI Processed Test Shape: {X_test_uci_prep.shape}")

C:\Users\mishr\AppData\Local\Temp\ipykernel_7416\809410307.py:2: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  uci_df = pd.read_csv(uci_path, na_values='?')


UCI Processed Train Shape: (81477, 20)
UCI Processed Test Shape: (20289, 20)
